# Current Coverage Rebound/Downtrend Router v1

Direction: `execution` / `selection` diagnostic.  
Role: `diagnose`, no runtime promotion from this notebook.  
Changed layer: fixed daily-position router on top of refreshed current-coverage paths.

## Governance

- Canonical baseline: scheduled/paper `balanced_rule_residual` family and current production coverage through May 2026.
- Direct comparison baseline: `current_coverage_residual_guard_refresh_v1` selected daily path, because it is the refreshed low-risk base after the stale residual-guard warning.
- Direct upside comparisons: recovery/re-entry execution paths with saved strict and walk-forward daily CSVs.
- Target window: strict OOS from `2020-01-02` through current coverage plus saved `wf_2020`, `wf_2022`, and `wf_2023_2024` windows.
- Accepted trade-off: a diagnostic may improve rebound participation, but it is not promotable unless strict OOS `relative_return >= +30pp` and `max_dd <= 15%`, with walk-forward guards.
- Comparison-table row key: `current_coverage_rebound_downtrend_router_v1`.

## Historical reflection

- Closest success: `risk_budget_residual_drawdown_guard_v1` and `state_conditioned_rebound_risk_router_v1` came close to the `+30pp / <=15%` strict target, but the former was stale through `2026-03-31` and the latter explicitly required refreshed coverage.
- Closest failure: `current_coverage_residual_guard_refresh_v1` found no full-coverage residual guard pass, and `poor_window_mechanism_router_v1` showed repeated 2025/2026 high-volatility rebound underparticipation after cap-like overlays.
- Why this is not a duplicate: this run starts from the refreshed current-coverage selected base, evaluates strict and walk-forward windows, and adds an explicit high-volatility rebound branch plus a separate high-volatility downtrend/SOXX-stress branch. It does not retune a single global cap.

## External basis

- Probability of Backtest Overfitting motivates a compressed, explicitly guarded grid instead of repeated threshold fishing: https://doi.org/10.21314/JCF.2016.322
- Deep Momentum Networks motivates separating trend estimation from position sizing and turnover-aware decisions: https://arxiv.org/abs/1904.04912
- Volatility-Managed Portfolios motivates volatility conditioning while this notebook checks for underparticipation risk: https://www.nber.org/papers/w22208
- Slow Momentum with Fast Reversion motivates separating slow trend/rebound exposure from fast adverse moves: https://arxiv.org/abs/2105.13727
- Qlib keeps the experiment in the repo-owned reproducible AI quant harness: https://github.com/microsoft/qlib

## Hypothesis

Persistent 2025/2026 bad windows are not primarily drawdown failures; they are high-volatility rebound underparticipation. A dual-state router should lift exposure only in high-volatility positive-rebound states while independently capping high-volatility downtrends or SOXL/SOXX relative stress.

Full pass requires:

- strict OOS `relative_return >= 30pp` and `max_dd <= 15%`;
- each saved walk-forward window `max_dd <= 15%`;
- walk-forward relative-return floors: `wf_2020 >= -10pp`, `wf_2022 >= 30pp`, `wf_2023_2024 >= 30pp`;
- 2025 and 2026 rebound clusters improve excess versus the refreshed base without breaching `15%` cluster MaxDD.


In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 240)


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "runtime").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")


ROOT = find_repo_root()
EXPERIMENT = "current_coverage_rebound_downtrend_router_v1"
CARD = ROOT / "runtime/execution/card" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

STARTING_NAV = 1_000_000.0
STRICT_TARGET_REL = 30.0
STRICT_TARGET_DD = 15.0
WF_REL_FLOORS = {"wf_2020": -10.0, "wf_2022": 30.0, "wf_2023_2024": 30.0}
WINDOWS = ["strict_oos", "wf_2020", "wf_2022", "wf_2023_2024"]
POOR_CLUSTERS = {
    "cluster_2025_high_vol_rebound": ("2025-04-16", "2025-07-17"),
    "cluster_2026_high_vol_rebound": ("2026-02-17", "2026-05-15"),
}

BASE_PATHS = {
    "strict_oos": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_strict_oos_daily.csv",
    "wf_2020": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_wf_2020_daily.csv",
    "wf_2022": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_wf_2022_daily.csv",
    "wf_2023_2024": ROOT / "runtime/execution/card/current_coverage_residual_guard_refresh_v1/selected_wf_2023_2024_daily.csv",
}

UPSIDE_FAMILIES = {
    "recovery_reentry_anchor_confirm_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/recovery_reentry_anchor_confirm_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2020/recovery_reentry_anchor_confirm_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2022/recovery_reentry_anchor_confirm_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_anchor_confirm_v1/walkforward/wf_2023_2024/recovery_reentry_anchor_confirm_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_conditional_vol_target_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/recovery_reentry_conditional_vol_target_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2020/recovery_reentry_conditional_vol_target_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2022/recovery_reentry_conditional_vol_target_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_conditional_vol_target_v1/walkforward/wf_2023_2024/recovery_reentry_conditional_vol_target_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_cppi_floor_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2020/recovery_reentry_cppi_floor_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2022/recovery_reentry_cppi_floor_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward/wf_2023_2024/recovery_reentry_cppi_floor_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_regime_drawdown_guard_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2020/recovery_reentry_regime_drawdown_guard_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2022/recovery_reentry_regime_drawdown_guard_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward/wf_2023_2024/recovery_reentry_regime_drawdown_guard_v1_wf_2023_2024_soxl_daily.csv",
    },
    "recovery_reentry_2021_trendbreak_cap_v1": {
        "strict_oos": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/recovery_reentry_2021_trendbreak_cap_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2020/recovery_reentry_2021_trendbreak_cap_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2022/recovery_reentry_2021_trendbreak_cap_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/recovery_reentry_2021_trendbreak_cap_v1/walkforward/wf_2023_2024/recovery_reentry_2021_trendbreak_cap_v1_wf_2023_2024_soxl_daily.csv",
    },
    "soxl_soxx_efficiency_chop_guard_v1": {
        "strict_oos": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/soxl_soxx_efficiency_chop_guard_v1_soxl_daily.csv",
        "wf_2020": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/walkforward/wf_2020/soxl_soxx_efficiency_chop_guard_v1_wf_2020_soxl_daily.csv",
        "wf_2022": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/walkforward/wf_2022/soxl_soxx_efficiency_chop_guard_v1_wf_2022_soxl_daily.csv",
        "wf_2023_2024": ROOT / "runtime/execution/card/soxl_soxx_efficiency_chop_guard_v1/walkforward/wf_2023_2024/soxl_soxx_efficiency_chop_guard_v1_wf_2023_2024_soxl_daily.csv",
    },
}

ROUTER_WINDOW_SOURCE_FAMILY = "recovery_reentry_anchor_confirm_v1"
ROUTER_FAMILY = "direct_current_coverage_base"
PANEL_PATH = ROOT / "data/processed/panels/soxl_soxx_vix_full.csv"

missing = [str(p.relative_to(ROOT)) for p in BASE_PATHS.values() if not p.exists()]
for family, paths in UPSIDE_FAMILIES.items():
    missing.extend(str(p.relative_to(ROOT)) for p in paths.values() if not p.exists())
if not PANEL_PATH.exists():
    missing.append(str(PANEL_PATH.relative_to(ROOT)))
if missing:
    raise FileNotFoundError("missing required paths:" + chr(10) + chr(10).join(missing))

print(f"repo={ROOT}")
print(f"card={CARD.relative_to(ROOT)}")
print(f"upside_families={len(UPSIDE_FAMILIES)}")


repo=/Users/rziiiii/.openclaw/workspace
card=runtime/execution/card/current_coverage_rebound_downtrend_router_v1
upside_families=6


In [2]:
def load_daily(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime").reset_index(drop=True)
    required = {"datetime", "price", "position"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.relative_to(ROOT)} missing {missing}")
    for col in df.columns:
        if col != "datetime" and df[col].dtype == object:
            df[col] = pd.to_numeric(df[col], errors="ignore")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df["position"] = pd.to_numeric(df["position"], errors="coerce").clip(0.0, 1.0)
    return df.dropna(subset=["price", "position"]).reset_index(drop=True)


def coalesce_feature(df: pd.DataFrame, names: list[str], default: float = np.nan) -> pd.Series:
    for name in names:
        if name in df.columns:
            return pd.to_numeric(df[name], errors="coerce")
    return pd.Series(default, index=df.index, dtype=float)


def load_semis_panel(path: Path) -> pd.DataFrame:
    panel = pd.read_csv(path)
    panel["datetime"] = pd.to_datetime(panel["date"])
    panel = panel.sort_values("datetime").reset_index(drop=True)
    soxl = pd.to_numeric(panel["soxl_close"], errors="coerce")
    soxx = pd.to_numeric(panel["soxx_close"], errors="coerce")
    ratio = soxl / soxx.replace(0.0, np.nan)
    ratio_mean = ratio.rolling(252, min_periods=63).mean()
    ratio_std = ratio.rolling(252, min_periods=63).std()
    out = pd.DataFrame({"datetime": panel["datetime"]})
    out["soxx_close"] = soxx
    out["soxx_mom_21"] = soxx / soxx.shift(21) - 1.0
    out["soxl_soxx_ratio"] = ratio
    out["soxl_soxx_ratio_mom_21"] = ratio / ratio.shift(21) - 1.0
    out["soxl_soxx_ratio_z_252"] = (ratio - ratio_mean) / ratio_std.replace(0.0, np.nan)
    out["soxl_soxx_ratio_widen_21"] = out["soxl_soxx_ratio_z_252"] - out["soxl_soxx_ratio_z_252"].shift(21)
    return out


SEMIS_PANEL = load_semis_panel(PANEL_PATH)


def recompute_nav(price: np.ndarray, position: np.ndarray) -> np.ndarray:
    returns = np.zeros_like(price, dtype=float)
    returns[1:] = price[1:] / price[:-1] - 1.0
    applied_position = np.r_[position[0], position[:-1]]
    return STARTING_NAV * np.cumprod(1.0 + applied_position * returns)


def metrics_from_position(price: np.ndarray, position: np.ndarray) -> dict[str, float]:
    nav = recompute_nav(price, position)
    buyhold = STARTING_NAV * price / price[0]
    strategy_return = (nav[-1] / nav[0] - 1.0) * 100.0
    buyhold_return = (buyhold[-1] / buyhold[0] - 1.0) * 100.0
    max_dd = (1.0 - nav / np.maximum.accumulate(nav)).max() * 100.0
    returns = price[1:] / price[:-1] - 1.0
    strat_returns = position[:-1] * returns if len(position) > 1 else np.array([], dtype=float)
    sharpe = float(np.mean(strat_returns) / np.std(strat_returns) * np.sqrt(252.0)) if len(strat_returns) > 2 and np.std(strat_returns) > 0 else 0.0
    turnover = np.abs(np.diff(position, prepend=position[0])).sum()
    return {
        "return": float(strategy_return),
        "buy_and_hold_return": float(buyhold_return),
        "relative_return": float(strategy_return - buyhold_return),
        "sharpe": sharpe,
        "max_dd": float(max_dd),
        "avg_pos": float(np.nanmean(position)),
        "turnover": float(turnover),
    }


def sub_metrics(window: dict, position: np.ndarray, start: str, end: str) -> dict[str, float]:
    dt = window["frame"]["datetime"]
    mask = (dt >= pd.Timestamp(start)) & (dt <= pd.Timestamp(end))
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) < 2:
        return {"return": np.nan, "buy_and_hold_return": np.nan, "relative_return": np.nan, "sharpe": np.nan, "max_dd": np.nan, "avg_pos": np.nan, "turnover": np.nan, "rows": int(len(idx))}
    m = metrics_from_position(window["price"][idx], position[idx])
    m["rows"] = int(len(idx))
    return m


def safe_quantile(values: np.ndarray, q: float) -> float:
    clean = values[np.isfinite(values)]
    if len(clean) == 0:
        return float("nan")
    return float(np.quantile(clean, q))


def build_window(base_path: Path, upside_path: Path, label: str) -> dict:
    base = load_daily(base_path)
    upside = load_daily(upside_path)
    keep = ["datetime", "position"]
    merged = base.merge(
        upside[keep].rename(columns={"position": "upside_position"}),
        on="datetime",
        how="inner",
        validate="one_to_one",
    )
    merged = merged.rename(columns={"position": "base_position"})
    merged = merged.loc[:, ~merged.columns.duplicated()]
    merged = merged.merge(SEMIS_PANEL, on="datetime", how="left")
    if merged.empty:
        raise ValueError(f"no overlap for {label}: {base_path} vs {upside_path}")
    alignment = {
        "base_rows": int(len(base)),
        "upside_rows": int(len(upside)),
        "merged_rows": int(len(merged)),
        "start": merged["datetime"].iloc[0].strftime("%Y-%m-%d"),
        "end": merged["datetime"].iloc[-1].strftime("%Y-%m-%d"),
        "coverage_truncated": bool(len(merged) != len(base) or len(merged) != len(upside)),
    }

    price_s = pd.to_numeric(merged["price"], errors="coerce")
    price = price_s.to_numpy(dtype=float)
    base_position = pd.to_numeric(merged["base_position"], errors="coerce").to_numpy(dtype=float)
    upside_position = pd.to_numeric(merged["upside_position"], errors="coerce").to_numpy(dtype=float)
    returns_s = price_s.pct_change()

    computed_mom5 = price_s / price_s.shift(5) - 1.0
    computed_mom10 = price_s / price_s.shift(10) - 1.0
    computed_mom21 = price_s / price_s.shift(21) - 1.0
    computed_dd63 = price_s / price_s.rolling(63, min_periods=2).max() - 1.0
    computed_rv21 = returns_s.rolling(21, min_periods=5).std() * np.sqrt(252.0)
    computed_rv_pct21 = computed_rv21.rolling(252, min_periods=63).rank(pct=True)
    computed_vov21 = computed_rv21.rolling(21, min_periods=5).std()

    features = {
        "mom5": coalesce_feature(merged, ["mom_5", "price_mom_5", "lag_mom_5"], np.nan).fillna(computed_mom5).fillna(0.0),
        "mom10": coalesce_feature(merged, ["mom_10", "price_mom_10", "lag_mom_10"], np.nan).fillna(computed_mom10).fillna(0.0),
        "mom21": coalesce_feature(merged, ["mom_21", "price_mom_21", "lag_mom_21"], np.nan).fillna(computed_mom21).fillna(0.0),
        "relative_strength_21": coalesce_feature(merged, ["relative_strength_21", "lag_relative_strength_21"], np.nan).fillna(0.0),
        "dd63": coalesce_feature(merged, ["drawdown_63", "price_drawdown_63", "lag_drawdown_63"], np.nan).fillna(computed_dd63).fillna(0.0),
        "rv21": coalesce_feature(merged, ["rv_21", "realized_vol_21", "lag_rv_21"], np.nan).fillna(computed_rv21).fillna(0.0),
        "rv_pct21": coalesce_feature(merged, ["realized_vol_pct_21", "lag_realized_vol_pct_21"], np.nan).fillna(computed_rv_pct21).fillna(0.0),
        "vol_of_vol_21": coalesce_feature(merged, ["vol_of_vol_21", "lag_vol_of_vol_21"], np.nan).fillna(computed_vov21).fillna(0.0),
        "vix_percentile_252": coalesce_feature(merged, ["vix_percentile_252", "lag_vix_percentile_252"], np.nan).fillna(0.0),
        "anchor_gap_180": coalesce_feature(merged, ["anchor_gap_180", "lag_anchor_gap_180"], np.nan).fillna(0.0),
        "below_anchor_ratio_63": coalesce_feature(merged, ["below_anchor_ratio_63", "lag_below_anchor_ratio_63"], np.nan).fillna(0.0),
        "soxx_mom21": coalesce_feature(merged, ["soxx_mom_21"], np.nan).fillna(0.0),
        "ratio_widen_21": coalesce_feature(merged, ["soxl_soxx_ratio_widen_21"], np.nan).fillna(0.0),
        "ratio_z_252": coalesce_feature(merged, ["soxl_soxx_ratio_z_252"], np.nan).fillna(0.0),
    }
    feature_arrays = {k: v.to_numpy(dtype=float) for k, v in features.items()}

    base_metrics = metrics_from_position(price, base_position)
    upside_metrics = metrics_from_position(price, upside_position)
    base_metrics.update(alignment)
    upside_metrics.update(alignment)
    return {
        "label": label,
        "frame": merged,
        "alignment": alignment,
        "price": price,
        "base_position": base_position,
        "upside_position": upside_position,
        "base_metrics": base_metrics,
        "upside_metrics": upside_metrics,
        **feature_arrays,
    }


def apply_trade_band(desired: np.ndarray, trade_band: float) -> np.ndarray:
    out = np.empty_like(desired, dtype=float)
    current = float(desired[0])
    for i, target in enumerate(desired):
        target = float(np.clip(target, 0.0, 1.0))
        if i > 0 and abs(target - current) < trade_band:
            target = current
        out[i] = target
        current = target
    return out


def router_position(window: dict, cfg: dict) -> tuple[np.ndarray, dict[str, np.ndarray | float]]:
    base = window["base_position"]
    desired = base.copy()
    high_vol = window["rv_pct21"] >= cfg["rebound_vol_pct"]
    trend_ok = (
        (window["mom21"] >= cfg["rebound_mom21"])
        & (window["mom10"] >= cfg["rebound_mom10"])
        & (window["dd63"] >= cfg["rebound_min_dd63"])
        & (window["relative_strength_21"] >= cfg["rebound_min_rs21"])
    )
    if cfg["require_anchor_recovery"]:
        trend_ok = trend_ok & (window["anchor_gap_180"] >= cfg["rebound_min_anchor_gap"])
    semis_ok = np.ones_like(base, dtype=bool)
    if cfg["require_soxx_positive"]:
        semis_ok = (window["soxx_mom21"] >= cfg["semis_soxx_mom21"]) & (window["ratio_widen_21"] >= cfg["semis_ratio_widen_min"])
    rebound = high_vol & trend_ok & semis_ok

    downtrend = high_vol & ((window["dd63"] <= cfg["risk_dd63"]) | (window["mom21"] <= cfg["risk_mom21"]))
    vix_stress = (window["vix_percentile_252"] >= cfg["risk_vix_pct"]) & (window["mom10"] <= cfg["risk_mom10"])
    semis_stress = ((window["ratio_z_252"] <= cfg["semis_ratio_z"]) | (window["ratio_widen_21"] <= cfg["semis_ratio_widen"])) & (window["soxx_mom21"] <= cfg["semis_soxx_mom21"])
    risk = downtrend | vix_stress | (semis_stress if cfg["use_semis_stress"] else False)

    desired[risk] = np.minimum(desired[risk], cfg["risk_cap"])
    rebound_target = np.maximum(base, cfg["rebound_target"])
    if cfg["use_upside_family_cap"]:
        rebound_target = np.minimum(np.maximum(rebound_target, window["upside_position"]), cfg["rebound_target"])
    desired[rebound & ~risk] = np.maximum(desired[rebound & ~risk], rebound_target[rebound & ~risk])
    position = apply_trade_band(np.clip(desired, 0.0, 1.0), cfg["trade_band"])
    state = {
        "rebound": rebound,
        "risk": risk,
        "high_vol": high_vol,
        "downtrend": downtrend,
        "vix_stress": vix_stress,
        "semis_stress": semis_stress,
    }
    return position, state


def score_candidate(metrics_by_window: dict[str, dict], cluster_by_name: dict[str, dict], base_cluster_by_name: dict[str, dict]) -> tuple[float, bool, bool, bool]:
    strict = metrics_by_window["strict_oos"]
    strict_pass = strict["relative_return"] >= STRICT_TARGET_REL and strict["max_dd"] <= STRICT_TARGET_DD
    wf_pass = True
    wf_loss = 0.0
    for wf, floor in WF_REL_FLOORS.items():
        m = metrics_by_window[wf]
        wf_pass = wf_pass and (m["max_dd"] <= STRICT_TARGET_DD) and (m["relative_return"] >= floor)
        wf_loss += 2.0 * max(0.0, m["max_dd"] - STRICT_TARGET_DD) ** 2
        wf_loss += 0.25 * max(0.0, floor - m["relative_return"]) ** 2
    strict_loss = max(0.0, STRICT_TARGET_REL - strict["relative_return"]) ** 2 + 4.0 * max(0.0, strict["max_dd"] - STRICT_TARGET_DD) ** 2
    cluster_pass = True
    cluster_loss = 0.0
    for name, m in cluster_by_name.items():
        base_m = base_cluster_by_name[name]
        excess_delta = m["relative_return"] - base_m["relative_return"]
        cluster_pass = cluster_pass and (excess_delta >= 0.0) and (m["max_dd"] <= STRICT_TARGET_DD)
        cluster_loss += 0.10 * max(0.0, 25.0 - excess_delta) ** 2
        cluster_loss += 2.0 * max(0.0, m["max_dd"] - STRICT_TARGET_DD) ** 2
    turnover_loss = 0.02 * strict["turnover"]
    return float(strict_loss + wf_loss + cluster_loss + turnover_loss), bool(strict_pass), bool(strict_pass and wf_pass), bool(strict_pass and wf_pass and cluster_pass)


In [3]:
windows_by_family: dict[str, dict[str, dict]] = {}
base_metrics_rows = []
upside_metrics_rows = []
for family, paths in UPSIDE_FAMILIES.items():
    windows_by_family[family] = {}
    for label in WINDOWS:
        window = build_window(BASE_PATHS[label], paths[label], label)
        windows_by_family[family][label] = window
        if family == next(iter(UPSIDE_FAMILIES)):
            base_metrics_rows.append({"family": "current_coverage_residual_guard_refresh_v1", "window": label, **window["base_metrics"]})
        upside_metrics_rows.append({"family": family, "window": label, **window["upside_metrics"]})

router_windows = windows_by_family[ROUTER_WINDOW_SOURCE_FAMILY]

base_metrics_df = pd.DataFrame(base_metrics_rows)
upside_metrics_df = pd.DataFrame(upside_metrics_rows)
base_metrics_df.to_csv(CARD / "base_metrics.csv", index=False)
upside_metrics_df.to_csv(CARD / "upside_metrics.csv", index=False)

display(Markdown("## Refreshed base metrics"))
display(base_metrics_df)
display(Markdown("## Upside family metrics"))
display(upside_metrics_df[["family", "window", "return", "relative_return", "max_dd", "avg_pos", "turnover", "start", "end"]].head(40))


/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is

/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")
/var/folders/4n/bc8v7c391nq228754jxg84_w0000gn/T/ipykernel_90905/762991931.py:11: FutureWarning: errors='ignore' is

## Refreshed base metrics

,family,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,turnover,base_rows,upside_rows,merged_rows,start,end,coverage_truncated
0,current_coverage_residual_guard_refresh_v1,strict_oos,691.382261,816.512500,-125.130239,1.052208,34.266345,0.316863,45.095227,1605,1605,1605,2020-01-02,2026-05-21,False
1,current_coverage_residual_guard_refresh_v1,wf_2020,71.525200,58.093571,13.431629,1.255681,24.815273,0.368370,7.605649,252,252,252,2020-01-02,2020-12-30,False
2,current_coverage_residual_guard_refresh_v1,wf_2022,-20.520014,-86.588072,66.068058,-0.741938,22.204418,0.161872,2.548362,250,250,250,2022-01-03,2022-12-29,False
3,current_coverage_residual_guard_refresh_v1,wf_2023_2024,67.254623,-0.356635,67.611258,1.678349,16.826953,0.258369,5.845345,251,251,251,2024-01-02,2024-12-30,False


## Upside family metrics

,family,window,return,relative_return,max_dd,avg_pos,turnover,start,end
0,recovery_reentry_anchor_confirm_v1,strict_oos,1894.611414,1078.098915,41.587207,0.466852,36.573357,2020-01-02,2026-05-21
1,recovery_reentry_anchor_confirm_v1,wf_2020,113.889962,55.796392,26.139320,0.423880,5.614635,2020-01-02,2020-12-30
2,recovery_reentry_anchor_confirm_v1,wf_2022,-20.683262,65.904809,26.651030,0.183541,2.819416,2022-01-03,2022-12-29
3,recovery_reentry_anchor_confirm_v1,wf_2023_2024,58.194746,58.551381,30.183184,0.336830,4.936599,2024-01-02,2024-12-30
4,recovery_reentry_conditional_vol_target_v1,strict_oos,2988.234589,2171.722089,30.923275,0.430109,37.983036,2020-01-02,2026-05-21
5,recovery_reentry_conditional_vol_target_v1,wf_2020,113.889962,55.796392,26.139320,0.423880,5.614635,2020-01-02,2020-12-30
6,recovery_reentry_conditional_vol_target_v1,wf_2022,-21.610018,64.978054,23.626625,0.165095,2.380487,2022-01-03,2022-12-29
7,recovery_reentry_conditional_vol_target_v1,wf_2023_2024,68.533647,68.890281,16.826953,0.259105,5.833511,2024-01-02,2024-12-30
8,recovery_reentry_cppi_floor_v1,strict_oos,2410.985976,1594.473476,32.120180,0.440279,38.601761,2020-01-02,2026-05-21
9,recovery_reentry_cppi_floor_v1,wf_2020,113.889962,55.796392,26.139320,0.423880,5.614635,2020-01-02,2020-12-30


In [4]:
GRID = {
    # Compressed after an initial oversized grid was too slow and too easy to overfit.
    # Kept axes are mechanism-level: rebound participation, downtrend cap,
    # SOXX stress usage, and trade absorption.
    "rebound_target": [0.45, 0.55, 0.65],
    "rebound_mom21": [0.25, 0.40],
    "rebound_mom10": [0.02],
    "rebound_vol_pct": [0.65, 0.80, 0.90],
    "rebound_min_dd63": [-0.25, -0.15],
    "rebound_min_rs21": [0.0],
    "require_anchor_recovery": [False, True],
    "require_soxx_positive": [False],
    "risk_cap": [0.00, 0.10, 0.20],
    "risk_dd63": [-0.08, -0.15],
    "risk_mom21": [0.0],
    "risk_vix_pct": [0.60, 0.75],
    "trade_band": [0.00, 0.03],
    "use_semis_stress": [False, True],
}
FIXED = {
    "risk_mom10": 0.0,
    "semis_ratio_z": -0.75,
    "semis_ratio_widen": -0.05,
    "semis_ratio_widen_min": -0.05,
    "semis_soxx_mom21": 0.0,
    "rebound_min_anchor_gap": -0.30,
    "use_upside_family_cap": False,
}


def iter_grid():
    keys = list(GRID)
    for values in product(*(GRID[k] for k in keys)):
        cfg = dict(zip(keys, values))
        cfg.update(FIXED)
        yield cfg


def compact_metrics(row: dict, prefix: str, metrics: dict) -> None:
    for k, v in metrics.items():
        row[f"{prefix}_{k}"] = v


rows = []
candidate_id = 0
base_cluster = {}
strict_ref = router_windows["strict_oos"]
for cluster_name, (start, end) in POOR_CLUSTERS.items():
    base_cluster[cluster_name] = sub_metrics(strict_ref, strict_ref["base_position"], start, end)

for cfg in iter_grid():
    metrics_by_window = {}
    state_counts = {}
    positions = {}
    for label, window in router_windows.items():
        position, state = router_position(window, cfg)
        positions[label] = position
        metrics_by_window[label] = metrics_from_position(window["price"], position)
        state_counts[label] = {f"{name}_days": int(np.sum(value)) for name, value in state.items()}
        state_counts[label]["changed_days"] = int(np.sum(np.abs(position - window["base_position"]) > 1e-9))
    strict_window = router_windows["strict_oos"]
    cluster_metrics = {
        name: sub_metrics(strict_window, positions["strict_oos"], start, end)
        for name, (start, end) in POOR_CLUSTERS.items()
    }
    robust_score, strict_pass, wf_pass, full_pass = score_candidate(metrics_by_window, cluster_metrics, base_cluster)
    row = {
        "candidate_id": candidate_id,
        "family": ROUTER_FAMILY,
        "window_source_family": ROUTER_WINDOW_SOURCE_FAMILY,
        **{k: cfg[k] for k in GRID.keys()},
        **FIXED,
        "robust_score": robust_score,
        "strict_pass": strict_pass,
        "wf_pass": wf_pass,
        "full_pass": full_pass,
    }
    for label in WINDOWS:
        compact_metrics(row, label, metrics_by_window[label])
        for k, v in state_counts[label].items():
            row[f"{label}_{k}"] = v
    for name, m in cluster_metrics.items():
        compact_metrics(row, name, m)
        row[f"{name}_excess_delta_vs_base"] = m["relative_return"] - base_cluster[name]["relative_return"]
        row[f"{name}_return_delta_vs_base"] = m["return"] - base_cluster[name]["return"]
    rows.append(row)
    candidate_id += 1

screen_df = pd.DataFrame(rows).sort_values(["full_pass", "wf_pass", "strict_pass", "robust_score"], ascending=[False, False, False, True]).reset_index(drop=True)
screen_df.to_csv(CARD / "router_screen.csv", index=False)
top_df = screen_df.head(100).copy()
top_df.to_csv(CARD / "router_top_configs.csv", index=False)
summary_df = (
    screen_df.groupby("family")
    .agg(
        candidates=("candidate_id", "count"),
        strict_pass_count=("strict_pass", "sum"),
        wf_pass_count=("wf_pass", "sum"),
        full_pass_count=("full_pass", "sum"),
        best_robust_score=("robust_score", "min"),
        best_strict_relative=("strict_oos_relative_return", "max"),
        best_strict_max_dd=("strict_oos_max_dd", "min"),
        best_2025_cluster_delta=("cluster_2025_high_vol_rebound_excess_delta_vs_base", "max"),
        best_2026_cluster_delta=("cluster_2026_high_vol_rebound_excess_delta_vs_base", "max"),
    )
    .reset_index()
    .sort_values(["full_pass_count", "wf_pass_count", "best_robust_score"], ascending=[False, False, True])
)
summary_df.to_csv(CARD / "router_family_summary.csv", index=False)
base_cluster_df = pd.DataFrame([{"cluster": k, **v} for k, v in base_cluster.items()])
base_cluster_df.to_csv(CARD / "base_cluster_metrics.csv", index=False)

print(f"screened_candidates={len(screen_df)}")
print(f"strict_pass_count={int(screen_df['strict_pass'].sum())}")
print(f"wf_pass_count={int(screen_df['wf_pass'].sum())}")
print(f"full_pass_count={int(screen_df['full_pass'].sum())}")
display(Markdown("## Family summary"))
display(summary_df)
display(Markdown("## Top router configs"))
display(top_df.head(12))


screened_candidates=3456
strict_pass_count=0
wf_pass_count=0
full_pass_count=0


## Family summary

,family,candidates,strict_pass_count,wf_pass_count,full_pass_count,best_robust_score,best_strict_relative,best_strict_max_dd,best_2025_cluster_delta,best_2026_cluster_delta
0,direct_current_coverage_base,3456,0,0,0,54163.42697,-197.306769,31.560899,1.765355,54.775393


## Top router configs

,candidate_id,family,window_source_family,rebound_target,rebound_mom21,rebound_mom10,rebound_vol_pct,rebound_min_dd63,rebound_min_rs21,require_anchor_recovery,require_soxx_positive,risk_cap,risk_dd63,risk_mom21,risk_vix_pct,trade_band,use_semis_stress,risk_mom10,semis_ratio_z,semis_ratio_widen,semis_ratio_widen_min,semis_soxx_mom21,rebound_min_anchor_gap,use_upside_family_cap,robust_score,strict_pass,wf_pass,full_pass,strict_oos_return,strict_oos_buy_and_hold_return,strict_oos_relative_return,strict_oos_sharpe,strict_oos_max_dd,strict_oos_avg_pos,strict_oos_turnover,strict_oos_rebound_days,strict_oos_risk_days,strict_oos_high_vol_days,strict_oos_downtrend_days,strict_oos_vix_stress_days,strict_oos_semis_stress_days,strict_oos_changed_days,wf_2020_return,wf_2020_buy_and_hold_return,wf_2020_relative_return,wf_2020_sharpe,wf_2020_max_dd,wf_2020_avg_pos,wf_2020_turnover,wf_2020_rebound_days,wf_2020_risk_days,wf_2020_high_vol_days,wf_2020_downtrend_days,wf_2020_vix_stress_days,wf_2020_semis_stress_days,wf_2020_changed_days,wf_2022_return,wf_2022_buy_and_hold_return,wf_2022_relative_return,wf_2022_sharpe,wf_2022_max_dd,wf_2022_avg_pos,wf_2022_turnover,wf_2022_rebound_days,wf_2022_risk_days,wf_2022_high_vol_days,wf_2022_downtrend_days,wf_2022_vix_stress_days,wf_2022_semis_stress_days,wf_2022_changed_days,wf_2023_2024_return,wf_2023_2024_buy_and_hold_return,wf_2023_2024_relative_return,wf_2023_2024_sharpe,wf_2023_2024_max_dd,wf_2023_2024_avg_pos,wf_2023_2024_turnover,wf_2023_2024_rebound_days,wf_2023_2024_risk_days,wf_2023_2024_high_vol_days,wf_2023_2024_downtrend_days,wf_2023_2024_vix_stress_days,wf_2023_2024_semis_stress_days,wf_2023_2024_changed_days,cluster_2025_high_vol_rebound_return,cluster_2025_high_vol_rebound_buy_and_hold_return,cluster_2025_high_vol_rebound_relative_return,cluster_2025_high_vol_rebound_sharpe,cluster_2025_high_vol_rebound_max_dd,cluster_2025_high_vol_rebound_avg_pos,cluster_2025_high_vol_rebound_turnover,cluster_2025_high_vol_rebound_rows,cluster_2025_high_vol_rebound_excess_delta_vs_base,cluster_2025_high_vol_rebound_return_delta_vs_base,cluster_2026_high_vol_rebound_return,cluster_2026_high_vol_rebound_buy_and_hold_return,cluster_2026_high_vol_rebound_relative_return,cluster_2026_high_vol_rebound_sharpe,cluster_2026_high_vol_rebound_max_dd,cluster_2026_high_vol_rebound_avg_pos,cluster_2026_high_vol_rebound_turnover,cluster_2026_high_vol_rebound_rows,cluster_2026_high_vol_rebound_excess_delta_vs_base,cluster_2026_high_vol_rebound_return_delta_vs_base
0,2590,direct_current_coverage_base,recovery_reentry_anchor_confirm_v1,0.65,0.25,0.02,0.8,-0.25,0.0,True,False,0.2,-0.15,0.0,0.75,0.03,False,0.0,-0.75,-0.05,-0.05,0.0,-0.3,False,54163.426970,False,False,False,619.205731,816.5125,-197.306769,1.011192,35.986375,0.317570,58.587567,48,422,317,255,263,585,296,42.333134,58.093571,-15.760437,0.939340,24.815273,0.351416,10.233788,0,45,3,3,42,58,25,-29.234207,-86.588072,57.353865,-1.382314,29.420535,0.159774,2.982126,5,144,111,101,90,159,87,54.422120,-0.356635,54.778755,1.460811,20.174799,0.239196,7.218608,6,107,87,81,53,103,70,43.998155,192.505345,-148.507190,4.369763,11.084311,0.484965,2.229789,63,0.152352,0.152352,69.361199,155.13597,-85.774771,5.335228,7.683821,0.311742,4.079926,63,51.364098,51.364098
1,2686,direct_current_coverage_base,recovery_reentry_anchor_confirm_v1,0.65,0.25,0.02,0.8,-0.15,0.0,True,False,0.2,-0.15,0.0,0.75,0.03,False,0.0,-0.75,-0.05,-0.05,0.0,-0.3,False,54163.426970,False,False,False,619.205731,816.5125,-197.306769,1.011192,35.986375,0.317570,58.587567,44,422,317,255,263,585,296,42.333134,58.093571,-15.760437,0.939340,24.815273,0.351416,10.233788,0,45,3,3,42,58,25,-29.234207,-86.588072,57.353865,-1.382314,29.420535,0.159774,2.982126,4,144,111,101,90,159,87,54.422120,-0.356635,54.778755,1.460811,20.174799,0.239196,7.218608,5,107,87,81,53,103,70,43.998155,192.505345,-148.507190,4.369763,11.084311,0.484965,2.229789,63,0.152352,0.152352,69.361199,155.13597,-85.774771,5.335228,7.683821,0.311742,4.

In [5]:
selected = screen_df.iloc[0].to_dict()
selected_family = str(selected["family"])
selected_candidate_id = int(selected["candidate_id"])
selected_cfg = {k: selected[k] for k in GRID.keys()}
selected_cfg.update(FIXED)

selected_metrics_rows = []
cluster_rows = []
for label, window in router_windows.items():
    position, state = router_position(window, selected_cfg)
    metrics = metrics_from_position(window["price"], position)
    selected_metrics_rows.append({
        "window": label,
        **metrics,
        **{f"{name}_days": int(np.sum(value)) for name, value in state.items()},
        "changed_days": int(np.sum(np.abs(position - window["base_position"]) > 1e-9)),
    })
    nav = recompute_nav(window["price"], position)
    buyhold = STARTING_NAV * window["price"] / window["price"][0]
    out = pd.DataFrame({
        "datetime": window["frame"]["datetime"].dt.strftime("%Y-%m-%d"),
        "price": window["price"],
        "strategy_nav": nav,
        "buyhold_nav": buyhold,
        "base_position": window["base_position"],
        "upside_position": window["upside_position"],
        "router_position": position,
        "position": position,
        "rebound_state": state["rebound"],
        "risk_state": state["risk"],
        "high_vol_state": state["high_vol"],
        "downtrend_state": state["downtrend"],
        "vix_stress_state": state["vix_stress"],
        "semis_stress_state": state["semis_stress"],
        "mom_10": window["mom10"],
        "mom_21": window["mom21"],
        "relative_strength_21": window["relative_strength_21"],
        "drawdown_63": window["dd63"],
        "realized_vol_pct_21": window["rv_pct21"],
        "realized_vol_21": window["rv21"],
        "vol_of_vol_21": window["vol_of_vol_21"],
        "vix_percentile_252": window["vix_percentile_252"],
        "anchor_gap_180": window["anchor_gap_180"],
        "below_anchor_ratio_63": window["below_anchor_ratio_63"],
        "soxx_mom_21": window["soxx_mom21"],
        "soxl_soxx_ratio_z_252": window["ratio_z_252"],
        "soxl_soxx_ratio_widen_21": window["ratio_widen_21"],
    })
    out_name = "selected_strict_oos_daily.csv" if label == "strict_oos" else f"selected_{label}_daily.csv"
    out.to_csv(CARD / out_name, index=False)

strict_window = router_windows["strict_oos"]
selected_pos, selected_state = router_position(strict_window, selected_cfg)
for cluster_name, (start, end) in POOR_CLUSTERS.items():
    selected_cluster = sub_metrics(strict_window, selected_pos, start, end)
    base_m = base_cluster[cluster_name]
    cluster_rows.append({
        "cluster": cluster_name,
        "start": start,
        "end": end,
        **{f"base_{k}": v for k, v in base_m.items()},
        **{f"selected_{k}": v for k, v in selected_cluster.items()},
        "excess_delta_vs_base": selected_cluster["relative_return"] - base_m["relative_return"],
        "return_delta_vs_base": selected_cluster["return"] - base_m["return"],
    })

selected_metrics_df = pd.DataFrame(selected_metrics_rows)
selected_metrics_df.to_csv(CARD / "selected_metrics.csv", index=False)
cluster_comparison_df = pd.DataFrame(cluster_rows)
cluster_comparison_df.to_csv(CARD / "selected_cluster_comparison.csv", index=False)

display(Markdown("## Selected metrics"))
display(selected_metrics_df)
display(Markdown("## Selected poor-cluster comparison"))
display(cluster_comparison_df)


## Selected metrics

,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,turnover,rebound_days,risk_days,high_vol_days,downtrend_days,vix_stress_days,semis_stress_days,changed_days
0,strict_oos,619.205731,816.512500,-197.306769,1.011192,35.986375,0.317570,58.587567,48,422,317,255,263,585,296
1,wf_2020,42.333134,58.093571,-15.760437,0.939340,24.815273,0.351416,10.233788,0,45,3,3,42,58,25
2,wf_2022,-29.234207,-86.588072,57.353865,-1.382314,29.420535,0.159774,2.982126,5,144,111,101,90,159,87
3,wf_2023_2024,54.422120,-0.356635,54.778755,1.460811,20.174799,0.239196,7.218608,6,107,87,81,53,103,70


## Selected poor-cluster comparison

,cluster,start,end,base_return,base_buy_and_hold_return,base_relative_return,base_sharpe,base_max_dd,base_avg_pos,base_turnover,base_rows,selected_return,selected_buy_and_hold_return,selected_relative_return,selected_sharpe,selected_max_dd,selected_avg_pos,selected_turnover,selected_rows,excess_delta_vs_base,return_delta_vs_base
0,cluster_2025_high_vol_rebound,2025-04-16,2025-07-17,43.845803,192.505345,-148.659542,4.353348,11.084311,0.485239,2.264245,63,43.998155,192.505345,-148.507190,4.369763,11.084311,0.484965,2.229789,63,0.152352,0.152352
1,cluster_2026_high_vol_rebound,2026-02-17,2026-05-15,17.997101,155.135970,-137.138869,2.576697,6.421473,0.181584,1.879926,63,69.361199,155.135970,-85.774771,5.335228,7.683821,0.311742,4.079926,63,51.364098,51.364098


In [6]:
def selected_metric(label: str) -> dict:
    row = selected_metrics_df.loc[selected_metrics_df["window"] == label].iloc[0].to_dict()
    return {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in row.items() if k != "window"}


def blockers_for(row: pd.Series | dict) -> list[str]:
    blockers = []
    if row["strict_oos_relative_return"] < STRICT_TARGET_REL:
        blockers.append(f"strict relative {row['strict_oos_relative_return']:.2f}pp < {STRICT_TARGET_REL:.2f}pp")
    if row["strict_oos_max_dd"] > STRICT_TARGET_DD:
        blockers.append(f"strict MaxDD {row['strict_oos_max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
    for wf, floor in WF_REL_FLOORS.items():
        if row[f"{wf}_relative_return"] < floor:
            blockers.append(f"{wf} relative {row[f'{wf}_relative_return']:.2f}pp < {floor:.2f}pp")
        if row[f"{wf}_max_dd"] > STRICT_TARGET_DD:
            blockers.append(f"{wf} MaxDD {row[f'{wf}_max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
    for _, cluster_row in cluster_comparison_df.iterrows():
        if cluster_row["selected_max_dd"] > STRICT_TARGET_DD:
            blockers.append(f"{cluster_row['cluster']} MaxDD {cluster_row['selected_max_dd']:.2f}% > {STRICT_TARGET_DD:.2f}%")
        if cluster_row["excess_delta_vs_base"] < 0.0:
            blockers.append(f"{cluster_row['cluster']} excess delta {cluster_row['excess_delta_vs_base']:.2f}pp < 0")
    return blockers

full_pass_count = int(screen_df["full_pass"].sum())
wf_pass_count = int(screen_df["wf_pass"].sum())
strict_pass_count = int(screen_df["strict_pass"].sum())
cluster_pass = bool((cluster_comparison_df["excess_delta_vs_base"] >= 0.0).all() and (cluster_comparison_df["selected_max_dd"] <= STRICT_TARGET_DD).all())
blockers = blockers_for(selected)

if full_pass_count > 0:
    decision = "diagnostic_candidate_for_training_followup"
    conclusion = [
        "At least one refreshed current-coverage router clears the strict and walk-forward screen.",
        "Do not promote directly from this notebook; implement the state-conditioned behavior in training/selection and rerun full promotion evidence.",
    ]
else:
    decision = "diagnostic_no_promotion"
    conclusion = [
        "No refreshed current-coverage rebound/downtrend router clears the full strict plus walk-forward plus cluster gate.",
        "This keeps the mechanism useful as a diagnostic but blocks runtime promotion.",
        "The selected router shows whether rebound participation can be improved without losing the 2022 defense; if blockers remain, the next step is training-time state-conditioned selection or actor refresh rather than another path-level threshold screen.",
        "Primary blockers: " + "; ".join(blockers[:8]),
    ]

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "role": "diagnose",
    "changed_layer": ["execution", "selection"],
    "candidate_count": int(len(screen_df)),
    "strict_pass_count": strict_pass_count,
    "wf_pass_count": wf_pass_count,
    "full_pass_count": full_pass_count,
    "cluster_pass": cluster_pass,
    "selected_candidate_id": selected_candidate_id,
    "selected_family": selected_family,
    "router_window_source_family": ROUTER_WINDOW_SOURCE_FAMILY,
    "selected_config": selected_cfg,
    "selected_metrics": {label: selected_metric(label) for label in WINDOWS},
    "selected_cluster_comparison": cluster_comparison_df.to_dict("records"),
    "selected_screen_row": {k: (float(v) if isinstance(v, (np.floating, float)) else int(v) if isinstance(v, (np.integer, int)) else v) for k, v in selected.items()},
    "acceptance": {
        "strict_relative_target_pp": STRICT_TARGET_REL,
        "strict_max_dd_target_pct": STRICT_TARGET_DD,
        "strict_pass": bool(selected["strict_pass"]),
        "wf_pass": bool(selected["wf_pass"]),
        "full_pass": bool(selected["full_pass"]),
        "cluster_pass": cluster_pass,
    },
    "external_sources": [
        "https://doi.org/10.21314/JCF.2016.322",
        "https://arxiv.org/abs/1904.04912",
        "https://www.nber.org/papers/w22208",
        "https://arxiv.org/abs/2105.13727",
        "https://github.com/microsoft/qlib",
    ],
    "conclusion": conclusion,
    "artifacts": {
        "notebook": str((CARD / f"{EXPERIMENT}.ipynb").relative_to(ROOT)),
        "base_metrics": str((CARD / "base_metrics.csv").relative_to(ROOT)),
        "upside_metrics": str((CARD / "upside_metrics.csv").relative_to(ROOT)),
        "base_cluster_metrics": str((CARD / "base_cluster_metrics.csv").relative_to(ROOT)),
        "router_screen": str((CARD / "router_screen.csv").relative_to(ROOT)),
        "router_top_configs": str((CARD / "router_top_configs.csv").relative_to(ROOT)),
        "router_family_summary": str((CARD / "router_family_summary.csv").relative_to(ROOT)),
        "selected_metrics": str((CARD / "selected_metrics.csv").relative_to(ROOT)),
        "selected_cluster_comparison": str((CARD / "selected_cluster_comparison.csv").relative_to(ROOT)),
        "selected_daily": {
            "strict_oos": str((CARD / "selected_strict_oos_daily.csv").relative_to(ROOT)),
            "wf_2020": str((CARD / "selected_wf_2020_daily.csv").relative_to(ROOT)),
            "wf_2022": str((CARD / "selected_wf_2022_daily.csv").relative_to(ROOT)),
            "wf_2023_2024": str((CARD / "selected_wf_2023_2024_daily.csv").relative_to(ROOT)),
        },
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
    },
}

with (CARD / "selection.json").open("w", encoding="utf-8") as f:
    json.dump(selection, f, indent=2, ensure_ascii=False)

print(json.dumps({
    "decision": decision,
    "selected_candidate_id": selected_candidate_id,
    "selected_family": selected_family,
    "router_window_source_family": ROUTER_WINDOW_SOURCE_FAMILY,
    "strict_pass_count": strict_pass_count,
    "wf_pass_count": wf_pass_count,
    "full_pass_count": full_pass_count,
    "cluster_pass": cluster_pass,
}, indent=2))


{
  "decision": "diagnostic_no_promotion",
  "selected_candidate_id": 2590,
  "selected_family": "direct_current_coverage_base",
  "router_window_source_family": "recovery_reentry_anchor_confirm_v1",
  "strict_pass_count": 0,
  "wf_pass_count": 0,
  "full_pass_count": 0,
  "cluster_pass": true
}


In [7]:
strict = selection["selected_metrics"]["strict_oos"]
wf2022 = selection["selected_metrics"]["wf_2022"]
wf2324 = selection["selected_metrics"]["wf_2023_2024"]
cluster_2025, cluster_2026 = selection["selected_cluster_comparison"]
run_conclusion = f"""
## Run conclusion

- Decision: `{selection['decision']}`; no runtime promotion.
- Screened candidates: `{selection['candidate_count']}` compressed mechanism-level configs; strict pass `{selection['strict_pass_count']}`, walk-forward pass `{selection['wf_pass_count']}`, full pass `{selection['full_pass_count']}`.
- Selected config: rebound target `{selection['selected_config']['rebound_target']:.2f}`, rebound mom21 `{selection['selected_config']['rebound_mom21']:.2f}`, vol pct `{selection['selected_config']['rebound_vol_pct']:.2f}`, risk cap `{selection['selected_config']['risk_cap']:.2f}`, trade band `{selection['selected_config']['trade_band']:.2f}`.
- Strict OOS: return `{strict['return']:.2f}%`, buy-hold `{strict['buy_and_hold_return']:.2f}%`, relative `{strict['relative_return']:.2f}pp`, MaxDD `{strict['max_dd']:.2f}%`, avg position `{strict['avg_pos']:.3f}`.
- Walk-forward blockers: `wf_2022` relative `{wf2022['relative_return']:.2f}pp` but MaxDD `{wf2022['max_dd']:.2f}%`; `wf_2023_2024` relative `{wf2324['relative_return']:.2f}pp` but MaxDD `{wf2324['max_dd']:.2f}%`.
- Poor clusters: 2025 excess delta `{cluster_2025['excess_delta_vs_base']:.2f}pp` with MaxDD `{cluster_2025['selected_max_dd']:.2f}%`; 2026 excess delta `{cluster_2026['excess_delta_vs_base']:.2f}pp` with MaxDD `{cluster_2026['selected_max_dd']:.2f}%`.
- Interpretation: path-level rebound lifting can repair the 2026 underparticipation cluster without breaking cluster drawdown, but it worsens strict OOS and keeps walk-forward MaxDD above `15%`. The next valid step is training-time state-conditioned selection or refreshed actor generation, not another path-level threshold/cap screen.
"""
display(Markdown(run_conclusion))



## Run conclusion

- Decision: `diagnostic_no_promotion`; no runtime promotion.
- Screened candidates: `3456` compressed mechanism-level configs; strict pass `0`, walk-forward pass `0`, full pass `0`.
- Selected config: rebound target `0.65`, rebound mom21 `0.25`, vol pct `0.80`, risk cap `0.20`, trade band `0.03`.
- Strict OOS: return `619.21%`, buy-hold `816.51%`, relative `-197.31pp`, MaxDD `35.99%`, avg position `0.318`.
- Walk-forward blockers: `wf_2022` relative `57.35pp` but MaxDD `29.42%`; `wf_2023_2024` relative `54.78pp` but MaxDD `20.17%`.
- Poor clusters: 2025 excess delta `0.15pp` with MaxDD `11.08%`; 2026 excess delta `51.36pp` with MaxDD `7.68%`.
- Interpretation: path-level rebound lifting can repair the 2026 underparticipation cluster without breaking cluster drawdown, but it worsens strict OOS and keeps walk-forward MaxDD above `15%`. The next valid step is training-time state-conditioned selection or refreshed actor generation, not another path-level threshold/cap screen.


## Notebook conclusion

The executable selection cell above is the source of truth for this run. Read `selection.json` and the selected metrics/cluster CSVs generated by this notebook for the final decision.
